# 117. 物流延期风险预测项目

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 32 / 34 步：把完整流程迁移到真实项目**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 用户消费价值预测项目  →  **本章任务：** 物流延期风险预测项目  →  **下一步：** 共享单车需求预测项目
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

在电商履约里，从客户下单到包裹签收，中间任何一个环节慢了，原本承诺的送达时间就会超期，客户体验和商家口碑都跟着受影响。



## 本章目标

学完本章，你将能够：

- **理解**：理解「物流延期风险预测项目」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「物流延期风险预测项目」的关键输出指标。
- **迁移**：能把「物流延期风险预测项目」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 117.1 数据字典

**背景引入**：在电商履约里，从客户下单到包裹签收，中间任何一个环节慢了，原本承诺的送达时间就会超期，客户体验和商家口碑都跟着受影响。本项目就用巴西电商平台 Olist 的数据，把订单、商品明细、客户和卖家四张表接起来，做一个能提前判断「哪些订单可能延期」的小系统。上手第一步就是先把每张表里「一行代表什么」搞清楚，这正是下面数据字典要帮你做的。

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| order_id | 订单主键 | 最终保持一单一行 |
| customer/seller_state | 客户州/卖家州 | 下单时可用类别特征 |
| promise_days | 承诺时长 | 预计送达日减下单日 |
| goods/freight_value | 商品额/运费 | 订单级数值特征 |
| late | 是否延期 | 实际签收晚于预计日 |

## 117.2 数据质量检查清单

- 订单主键及明细一对多关系
- 聚合连接后订单唯一
- 日期缺失与承诺天数异常
- 已签收样本的选择口径
- 实际发货和签收字段泄漏（打个比方：预测“会不会延期”，不能用“已经签收”的信息当特征——就像比赛没开打就拿着比分来推测，当然“准”得没意义。）
- 延期率随时间和地区变化


## 117.3 项目任务

1. 明确订单粒度、预测时点与延期标签
2. 审计四张原始数据表
3. 聚合明细并连接订单级样本
4. 清洗日期并构造延期标签
5. 探索类别不平衡和场景差异
6. 审计泄漏并按时间划分三组数据
7. 建立预处理Pipeline和概率基线
8. 比较逻辑回归与随机森林
9. 评价PR-AUC、阈值、Lift与错误切片
10. 解释特征重要性并总结局限


## 117.4 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 加载四表并审计数据粒度 | `pd.read_csv()`、`pd.Series()`、`orders.order_id.duplicated()`、`items.order_id.nunique()` | 先确认每张表的一行代表什么，再决定聚合和连接方式。 | 订单主键及明细一对多关系 |
| 2. 聚合明细并连接订单级样本 | `items.groupby()`、`orders.merge()`、`order_level.order_delivered_customer_date.isna()`、`.agg()` | 先把一对多商品明细聚合到订单，再用validate检查连接基数。 | 聚合连接后订单唯一 |
| 3. 清洗样本并定义延期标签 | `order_level.order_status.eq()`、`order_level.order_delivered_customer_date.notna()`、`order_level.order_estimated_delivery_date.notna()`、`dt.total_seconds()` | 标签来自签收结果；建模样本限定为有完整日期的已签收订单，并记录保留率。 | 日期缺失与承诺天数异常 |
| 4. 探索延期率与类别不平衡 | `pd.qcut()`、`model_df.groupby()`、`late.agg()`、`month_rate.round()` | 观察月份、承诺时长和订单规模的差异，但不把描述性相关解释为延期原因。 | 已签收样本的选择口径 |
| 5. 泄漏审计与时间顺序划分 | `part.late.mean()`、`train.order_purchase_timestamp.max()`、`test.order_purchase_timestamp.min()`、`iloc[:train_end]` | 用早期订单训练、中期订单验证、晚期订单测试，模拟模型面对未来数据。 | 实际发货和签收字段泄漏 |
| 6. 预处理Pipeline与概率基线 | `dummy.predict_proba()`、`val.late.mean()`、`.fit()`、`train[features]` | 类别编码、数值缩放和模型封装为统一流程；Dummy概率作为最低基线。 | 延期率随时间和地区变化 |
| 7. 比较逻辑回归与随机森林 | `models.items()`、`model.fit()`、`rows.append()`、`model.predict_proba()` | 在同一验证集上比较两个常见分类器，测试集仍保持未查看。 | 订单主键及明细一对多关系 |
| 8. 测试集概率指标与阈值比较 | `pd.Series()`、`pd.DataFrame()`、`test.late.to_numpy()`、`ranked.head()` | PR-AUC是主指标，并比较不同Top-K比例下的精确率、召回率和Lift。 | 聚合连接后订单唯一 |
| 9. 错误类型与地区切片 | `np.select()`、`error_df.groupby()`、`error_df.error_type.value_counts()`、`state_report.head()` | 区分漏判与误报，并检查模型在主要客户州的错误率是否一致。 | 日期缺失与承诺天数异常 |
| 10. 特征解释与模型局限 | `np.linspace()`、`pd.Series()`、`importance.round()`、`.sort_values()` | 用测试子样本计算置换重要性，并讨论数据缺失和预测边界。 | 已签收样本的选择口径 |


## 117.5 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 117.6 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 117.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 117.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 117.9 加载四表并审计数据粒度

先确认每张表的一行代表什么，再决定聚合和连接方式。


<!-- math-foundation:chapter-117 -->
### 数学推导｜上线决策应最小化预期错误成本

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜阈值决定三种业务数量。** 对每个 $t$，统计误报 $FP(t)$、漏报 $FN(t)$ 与触发行动数 $A(t)=TP(t)+FP(t)$。

**第 2 步｜把数量换成同一货币或效用单位。** 

$$
C(t)=c_{FP}FP(t)+c_{FN}FN(t)+c_AA(t)
$$

**第 3 步｜比较候选阈值。** 在验证集上求 $t^*=\arg\min_tC(t)$，并同时检查容量约束 $A(t)\le A_{max}$。

**第 4 步｜锁定方案后再评估。** 测试集只用于估计已锁定 $t^*$ 的成本与波动，不能继续用它调阈值。

**把上面的关系收束为本章计算式：**

$$
C(t)=c_{FP}FP(t)+c_{FN}FN(t)+c_AA(t),\qquad t^*=\arg\min_t C(t)
$$

**符号解释：** $FP(t)$、$FN(t)$ 是阈值 $t$ 下两类错误数，$A(t)$ 是行动量；$c_{FP}$、$c_{FN}$、$c_A$ 是对应单位成本。

**代码对应：** 在验证集比较阈值下的错误、行动量与成本，再锁定阈值评估测试集。

**使用边界：** 成本估计也有不确定性；上线结论应包含试点、监控和停止条件。


In [ ]:
import numpy as np
import pandas as pd

orders = pd.read_csv(
    "/datasets/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "order_delivered_carrier_date",
    ],
)
items = pd.read_csv("/datasets/olist_order_items_dataset.csv")
customers = pd.read_csv("/datasets/olist_customers_dataset.csv")
sellers = pd.read_csv("/datasets/olist_sellers_dataset.csv")
audit = pd.Series(
    {
        "订单行": len(orders),
        "订单键重复": orders.order_id.duplicated().sum(),
        "明细行": len(items),
        "明细订单数": items.order_id.nunique(),
        "客户键重复": customers.customer_id.duplicated().sum(),
        "卖家键重复": sellers.seller_id.duplicated().sum(),
    }
)
print(audit.to_string())


**练一练**：审计订单明细的一对多关系，确认每个订单对应几行商品明细。

数据加载只是开始，真正要落实的是「一行代表什么」。本项目里 `items` 表按商品明细逐行记录，`orders` 表一单一行；只有先把这种一对多关系数清楚，后面的聚合与连接才不会被重复行悄悄放大。下方用一段与 olist 同口径的内置示例数据模拟两张表的粒度——若你能连到真实的 `/datasets/olist_*.csv`，就用同样思路对真实 `orders` 与 `items` 各跑一遍审计。


In [ ]:
import pandas as pd

# 内置示例数据：A01 有 2 行明细、A03 有 3 行明细，order_id 存在重复（一对多）
# 请在下方填写代码：按 order_id 统计每单的商品明细行数（提示：groupby + size），
# 结果存到 item_line_count，并打印出来
# item_line_count = sample_items.groupby("order_id").size()   # <-- 补全并取消注释
# print(item_line_count)


## 117.10 聚合明细并连接订单级样本

先把一对多商品明细聚合到订单，再用validate检查连接基数。


In [ ]:
item_agg = items.groupby("order_id").agg(
    item_count=("order_item_id", "size"),
    goods_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
    primary_seller=("seller_id", "first"),
    seller_count=("seller_id", "nunique"),
)
order_level = (
    orders.merge(item_agg, on="order_id", validate="one_to_one")
    .merge(
        customers[["customer_id", "customer_state"]],
        on="customer_id",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]].rename(
            columns={"seller_id": "primary_seller"}
        ),
        on="primary_seller",
        validate="many_to_one",
    )
)
_check_1 = bool(order_level.order_id.is_unique)
print("自检 1：order_level.order_id.is_unique ->", "通过" if _check_1 else "需要检查")
if not _check_1:
    print("建议：", "请回看输入、处理步骤和预期结果。")
print(
    "连接后订单:",
    len(order_level),
    "缺少实际签收:",
    order_level.order_delivered_customer_date.isna().sum(),
)
display(
    order_level[
        [
            "order_id",
            "item_count",
            "goods_value",
            "freight_value",
            "seller_count",
        ]
    ].head()
)


## 117.11 清洗样本并定义延期标签

标签来自签收结果；建模样本限定为有完整日期的已签收订单，并记录保留率。


In [ ]:
complete = (
    order_level.order_status.eq("delivered")
    & order_level.order_delivered_customer_date.notna()
    & order_level.order_estimated_delivery_date.notna()
)
model_df = order_level.loc[complete].copy()
model_df["promise_days"] = (
    model_df.order_estimated_delivery_date - model_df.order_purchase_timestamp
).dt.total_seconds() / 86400
model_df = model_df.query("promise_days>0")
model_df["late"] = (
    model_df.order_delivered_customer_date
    > model_df.order_estimated_delivery_date
).astype(int)
model_df["month"] = model_df.order_purchase_timestamp.dt.month
model_df["weekday"] = model_df.order_purchase_timestamp.dt.dayofweek
model_df = model_df.sort_values("order_purchase_timestamp").reset_index(
    drop=True
)
print(
    "建模订单:",
    len(model_df),
    "保留率:",
    f"{len(model_df)/len(orders):.1%}",
    "延期率:",
    f"{model_df.late.mean():.2%}",
)


## 117.12 探索延期率与类别不平衡

观察月份、承诺时长和订单规模的差异，但不把描述性相关解释为延期原因。


In [ ]:
model_df["promise_group"] = pd.qcut(
    model_df.promise_days, 4, duplicates="drop"
)
month_rate = model_df.groupby("month").late.agg(["size", "mean"])
promise_rate = model_df.groupby("promise_group", observed=True).late.agg(
    ["size", "mean"]
)
print("月份延期率:\n", month_rate.round(3))
print("承诺时长分组延期率:\n", promise_rate.round(3))
print("多数类准确率:", f"{max(model_df.late.mean(),1-model_df.late.mean()):.2%}")


## 117.13 泄漏审计与时间顺序划分

用早期订单训练、中期订单验证、晚期订单测试，模拟模型面对未来数据。


In [ ]:
num = [
    "item_count",
    "goods_value",
    "freight_value",
    "seller_count",
    "month",
    "weekday",
    "promise_days",
]
cat = ["customer_state", "seller_state"]
features = num + cat
forbidden = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_status",
]
assert not set(features) & set(forbidden)
train_end = int(len(model_df) * 0.64)
val_end = int(len(model_df) * 0.80)
train = model_df.iloc[:train_end]
val = model_df.iloc[train_end:val_end]
test = model_df.iloc[val_end:]
print("禁止字段:", forbidden)
print("训练/验证/测试:", len(train), len(val), len(test))
print("延期率:", *[f"{part.late.mean():.2%}" for part in [train, val, test]])
print(
    "训练截止:",
    train.order_purchase_timestamp.max(),
    "测试开始:",
    test.order_purchase_timestamp.min(),
)


## 117.14 预处理Pipeline与概率基线

类别编码、数值缩放和模型封装为统一流程；Dummy概率作为最低基线。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ("num", StandardScaler(), num),
    ]
)
dummy = DummyClassifier(strategy="prior").fit(train[features], train.late)
dummy_prob = dummy.predict_proba(val[features])[:, 1]
print(
    "验证集正类率:",
    round(val.late.mean(), 3),
    "Dummy PR-AUC:",
    round(average_precision_score(val.late, dummy_prob), 3),
)


## 117.15 比较逻辑回归与随机森林

在同一验证集上比较两个常见分类器，测试集仍保持未查看。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "逻辑回归": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                LogisticRegression(max_iter=700, class_weight="balanced"),
            ),
        ]
    ),
    "随机森林": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=160,
                    min_samples_leaf=8,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=106,
                ),
            ),
        ]
    ),
}
rows = []
for name, model in models.items():
    model.fit(train[features], train.late)
    rows.append(
        [
            name,
            average_precision_score(
                val.late, model.predict_proba(val[features])[:, 1]
            ),
        ]
    )
validation = pd.DataFrame(
    rows, columns=["model", "validation_PR_AUC"]
).sort_values("validation_PR_AUC", ascending=False)
display(validation.round(3))
best_name = validation.iloc[0].model
dev = model_df.iloc[:val_end]
best_model = models[best_name].fit(dev[features], dev.late)
probability = best_model.predict_proba(test[features])[:, 1]


## 117.16 测试集概率指标与阈值比较

PR-AUC是主指标，并比较不同Top-K比例下的精确率、召回率和Lift。


In [ ]:
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix

metrics = pd.Series(
    {
        "ROC_AUC": roc_auc_score(test.late, probability),
        "PR_AUC": average_precision_score(test.late, probability),
        "LogLoss": log_loss(test.late, probability),
    }
)
ranked = pd.DataFrame(
    {"actual": test.late.to_numpy(), "probability": probability}
).sort_values("probability", ascending=False)
threshold_rows = []
for share in [0.05, 0.10, 0.20]:
    n = max(1, int(len(ranked) * share))
    top = ranked.head(n)
    threshold_rows.append(
        [
            f"{share:.0%}",
            top.probability.min(),
            top.actual.mean(),
            top.actual.sum() / ranked.actual.sum(),
            top.actual.mean() / ranked.actual.mean(),
        ]
    )
threshold_table = pd.DataFrame(
    threshold_rows, columns=["Top比例", "概率阈值", "Precision", "Recall", "Lift"]
)
print(metrics.round(3).to_string())
display(threshold_table.round(3))
threshold = threshold_table.loc[
    threshold_table["Top比例"] == "10%", "概率阈值"
].iloc[0]
prediction = probability >= threshold
print("Top10%混淆矩阵:", confusion_matrix(test.late, prediction).tolist())


## 117.17 错误类型与地区切片

区分漏判与误报，并检查模型在主要客户州的错误率是否一致。


In [ ]:
error_df = test[
    ["customer_state", "promise_days", "item_count", "late"]
].copy()
error_df["probability"] = probability
error_df["prediction"] = prediction
error_df["error_type"] = np.select(
    [
        (error_df.late == 1) & (~error_df.prediction),
        (error_df.late == 0) & error_df.prediction,
    ],
    ["漏判延期", "误报延期"],
    default="判断正确",
)
state_report = (
    error_df.groupby("customer_state")
    .agg(
        orders=("late", "size"),
        late_rate=("late", "mean"),
        mean_score=("probability", "mean"),
        error_rate=("error_type", lambda x: (x != "判断正确").mean()),
    )
    .query("orders>=200")
    .sort_values("error_rate", ascending=False)
)
print(error_df.error_type.value_counts())
display(state_report.head(12).round(3))


## 117.18 特征解释与模型局限

用测试子样本计算置换重要性，并讨论数据缺失和预测边界。


In [ ]:
from sklearn.inspection import permutation_importance

sample_n = min(4000, len(test))
sample_idx = np.linspace(0, len(test) - 1, sample_n, dtype=int)
permutation = permutation_importance(
    best_model,
    test.iloc[sample_idx][features],
    test.iloc[sample_idx].late,
    n_repeats=3,
    scoring="average_precision",
    random_state=106,
    n_jobs=-1,
)
importance = pd.Series(
    permutation.importances_mean, index=features
).sort_values(ascending=False)
print("最佳模型:", best_name)
print("置换重要性:\n", importance.round(4))
print("局限: 数据缺少距离、仓库节点、承运商和实时轨迹；地区特征的重要性不能解释为地区导致延期。")


## 117.19 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 117.19.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 117.19.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 117.20 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 117.20.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 117.21 易错点提醒

**易错点 1**：延期标签口径要统一（晚于承诺时间即延期？晚多少小时算延期？），先定死再建标签，否则模型目标漂移。

**易错点 2**：特征只能用"预测时点之前"的信息（下单、付款、发货状态），不能用签收时间算出来的量。

**易错点 3**：取消单与缺货单不属于"按时/延期"二分，建模前先过滤或单独成类。

**易错点 4**：四张表连接后行数翻倍（商品明细多行）时，先确定预测粒度是订单级还是商品级。

**易错点 5**：时区不统一导致"承诺时间"与"签收时间"比较出错，先统一到同一时区再算时长。


## 117.22 结论与表达

- 多表建模必须先统一到订单粒度
- 标签可使用事后结果，但特征必须在预测时可获得
- 时间顺序划分比随机划分更接近未来预测
- 类别不平衡任务应联合观察PR-AUC、Recall和Lift


## 117.23 项目验收清单

- 完成四表粒度与连接审计
- 订单主键唯一且记录清洗口径
- 排除实际发货、签收和最终状态字段
- 比较Dummy和两个候选模型
- 完成阈值、错误切片和置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 117.24 小结

使用 Olist 巴西电商公开数据，以物流延期分类为主线，学习多表建模、预测时点、数据泄漏、类别不平衡和业务阈值。


### 117.24.1 你已经完成

- 理解订单、明细、客户和卖家表的粒度
- 构造订单级延期标签并排除事后字段
- 使用时间顺序划分模拟未来预测
- 比较概率基线、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片与特征重要性评价模型


### 117.24.2 质量与结论提醒

- 订单主键及明细一对多关系
- 聚合连接后订单唯一
- 日期缺失与承诺天数异常
- 多表建模必须先统一到订单粒度
- 标签可使用事后结果，但特征必须在预测时可获得
- 时间顺序划分比随机划分更接近未来预测
- 类别不平衡任务应联合观察PR-AUC、Recall和Lift


### 117.24.3 学习检查

- [ ] 完成四表粒度与连接审计
- [ ] 订单主键唯一且记录清洗口径
- [ ] 排除实际发货、签收和最终状态字段
- [ ] 比较Dummy和两个候选模型
- [ ] 完成阈值、错误切片和置换重要性分析


### 117.24.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
